⚡ Intermédiaire | ⏱ 60 min | 🔑 interface publique, `_protégé`, `__privé`, `@property`, validation, invariant

# 03 - Encapsulation

À la fin : tu sais décider ce qu'un objet expose et ce qu'il cache, protéger son état avec des méthodes et des propriétés, et garantir qu'il reste cohérent.


## 1. L'idée

**Encapsuler**, c'est regrouper données et méthodes dans l'objet, puis **cacher** ce qui est interne pour n'**exposer** qu'une interface publique. Tu conduis avec le volant et les pédales, pas en touchant à l'injection.

Pourquoi : un attribut public peut être mis dans n'importe quel état par n'importe qui. Une méthode, elle, peut **valider**.


In [ ]:
# Sans encapsulation : rien n'empêche un état absurde
class Compte:
    def __init__(self, solde):
        self.solde = solde

c = Compte(100)
c.solde = -1_000_000           # personne ne vérifie
print(c.solde)


In [ ]:
# Avec encapsulation : l'attribut devient interne, la méthode valide
class Compte:
    def __init__(self, solde):
        self._solde = solde

    def retirer(self, montant):
        if montant <= 0 or montant > self._solde:
            raise ValueError("retrait impossible")
        self._solde -= montant

    def solde(self):
        return self._solde

c = Compte(100)
c.retirer(30)
print(c.solde())


## 2. Visibilité : les conventions Python

| Écriture | Visibilité | Sens |
|---|---|---|
| `nom` | public | fait partie de l'interface : les autres peuvent l'utiliser, on s'engage à ne pas le casser |
| `_nom` | protégé (convention) | interne : « ne touche pas, sauf si tu sais ce que tu fais » |
| `__nom` | privé (*name mangling*) | Python le renomme `_Classe__nom` pour éviter les collisions dans l'héritage |

Python ne bloque rien (« *we're all consenting adults* »). Le `_` est un contrat entre développeurs. En UML : `+` public, `#` protégé, `-` privé.


In [ ]:
# _attribut : accessible, mais c'est un signal « interne, ne pas utiliser depuis l'extérieur »
print(c._solde)                # possible, déconseillé


In [ ]:
# __attribut : renommé par Python, inaccessible sous son nom d'origine
class Carte:
    def __init__(self, code):
        self.__pin = code

carte = Carte("1234")
try:
    print(carte.__pin)
except AttributeError as e:
    print("AttributeError :", e)
print(vars(carte))             # le vrai nom : _Carte__pin


In [ ]:
# Une méthode interne commence aussi par _ : utilisée par la classe, pas par l'extérieur
class Compte:
    def __init__(self, solde):
        self._solde = solde
        self._historique = []

    def _enregistrer(self, libelle, montant):      # interne
        self._historique.append((libelle, montant))

    def deposer(self, montant):                    # public
        self._solde += montant
        self._enregistrer("dépôt", montant)

c = Compte(100)
c.deposer(50)
print(c._historique)


## 3. `@property` : un attribut contrôlé

Les *getters* et *setters* à la Java (`get_solde()`, `set_solde(x)`) sont lourds à lire. `@property` garde la **syntaxe d'un attribut** (`c.solde`, `c.solde = 10`) tout en passant par une méthode. Trois usages : lecture seule, validation à l'écriture, valeur calculée.


In [ ]:
# @property : lecture via c.solde (sans parenthèses), écriture impossible
class Compte:
    def __init__(self, solde):
        self._solde = solde

    @property
    def solde(self):
        return self._solde

c = Compte(100)
print(c.solde)
try:
    c.solde = 0
except AttributeError as e:
    print("AttributeError :", e)


In [ ]:
# @solde.setter : l'écriture passe par une validation, y compris depuis __init__
class Compte:
    def __init__(self, solde):
        self.solde = solde             # appelle déjà le setter

    @property
    def solde(self):
        return self._solde

    @solde.setter
    def solde(self, valeur):
        if valeur < 0:
            raise ValueError("le solde ne peut pas être négatif")
        self._solde = valeur

c = Compte(100)
c.solde = 250
print(c.solde)
try:
    c.solde = -1
except ValueError as e:
    print("ValueError :", e)


In [ ]:
# Propriété calculée : rien n'est stocké, la valeur est toujours à jour
class Rectangle:
    def __init__(self, largeur, hauteur):
        self.largeur = largeur
        self.hauteur = hauteur

    @property
    def aire(self):
        return self.largeur * self.hauteur

r = Rectangle(2, 3)
r.largeur = 10
print(r.aire)


In [ ]:
# Deux propriétés, une seule donnée stockée : les conversions restent cohérentes
class Temperature:
    def __init__(self, celsius):
        self._celsius = celsius

    @property
    def celsius(self):
        return self._celsius

    @property
    def fahrenheit(self):
        return self._celsius * 9 / 5 + 32

    @fahrenheit.setter
    def fahrenheit(self, valeur):
        self._celsius = (valeur - 32) * 5 / 9

t = Temperature(25)
t.fahrenheit = 212
print(t.celsius, t.fahrenheit)


## 4. Invariants

Un **invariant** est une règle toujours vraie pour un objet valide : `solde >= 0`, `largeur > 0`, `0 <= batterie <= 100`. L'encapsulation sert à le **garantir** : on valide dans `__init__`, dans les setters et dans chaque méthode qui modifie l'état. Si l'état est faux, le bug vient forcément d'une méthode de la classe : on sait où chercher.


In [ ]:
# Invariant 0 <= batterie <= 100 : toutes les modifications passent par un seul endroit
class Telephone:
    def __init__(self, batterie=100):
        self.batterie = batterie                      # validé par le setter

    @property
    def batterie(self):
        return self._batterie

    @batterie.setter
    def batterie(self, valeur):
        self._batterie = max(0, min(100, valeur))     # bornage

    def utiliser(self, heures):
        self.batterie -= 10 * heures                  # repasse par le setter

tel = Telephone()
tel.utiliser(12)
print(tel.batterie)


## 5. Ne pas laisser fuir l'état interne

Renvoyer une liste interne, c'est donner les clés de la maison : l'appelant peut la modifier sans passer par la classe. Renvoyer une **copie** ou un **tuple**.


In [ ]:
# Fuite : la liste interne est modifiable depuis l'extérieur
class Panier:
    def __init__(self):
        self._articles = []

    def ajouter(self, article):
        self._articles.append(article)

    def articles(self):
        return self._articles             # ❌ référence directe

p = Panier()
p.ajouter("pain")
p.articles().clear()                      # vide le panier sans passer par la classe
print(p.articles())


In [ ]:
# Correction : renvoyer un tuple (immuable) ou une copie
class Panier:
    def __init__(self):
        self._articles = []

    def ajouter(self, article):
        self._articles.append(article)

    @property
    def articles(self):
        return tuple(self._articles)      # ✅ copie immuable

p = Panier()
p.ajouter("pain")
print(p.articles, len(p.articles))


## 6. Parler à l'interface, pas aux entrailles

**Couplage faible** : une classe utilise les **méthodes publiques** d'une autre, jamais ses attributs `_`. Si l'implémentation change, le code qui l'utilise ne casse pas.


In [ ]:
# ❌ Couplage fort : Station dépend du nom interne _carburant de Voiture
class Voiture:
    def __init__(self):
        self._carburant = 10

class Station:
    def plein(self, voiture):
        voiture._carburant = 50           # si Voiture renomme l'attribut, Station casse

v = Voiture()
Station().plein(v)
print(v._carburant)


In [ ]:
# ✅ Couplage faible : Station n'utilise que l'interface publique de Voiture
class Voiture:
    def __init__(self):
        self._carburant = 10

    def remplir(self, litres):
        self._carburant = min(50, self._carburant + litres)

    @property
    def carburant(self):
        return self._carburant

class Station:
    def plein(self, voiture):
        voiture.remplir(50)

v = Voiture()
Station().plein(v)
print(v.carburant)


## 7. Quand encapsuler ?

Règle : **commencer public**, simple. Passer à `_attribut` + `@property` quand il y a une vraie raison : validation, invariant, valeur calculée, lecture seule. Une classe pleine de getters et setters qui ne font rien n'apporte rien de plus qu'un dict.


In [ ]:
# Inutile : une property qui ne fait rien de plus qu'un attribut
class Point:
    def __init__(self, x):
        self._x = x

    @property
    def x(self):
        return self._x

    @x.setter
    def x(self, valeur):
        self._x = valeur

# Suffisant : attribut public ; on ajoutera une property le jour où une règle apparaît
class Point:
    def __init__(self, x):
        self.x = x

print(Point(3).x)


## ⚠️ Pièges

- `_x` n'est pas protégé par Python, `__x` non plus (`obj._Classe__x`). Convention, pas sécurité.
- Oublier le setter : `obj.prop = 1` lève `AttributeError: property 'prop' of 'X' object has no setter`.
- Donner le même nom à l'attribut de stockage et à la property (`return self.solde` dans le getter `solde`) : récursion infinie. Le stockage s'appelle `_solde`.
- Contourner son propre setter : écrire `self._solde = x` dans `__init__` alors qu'une règle existe. Passer par `self.solde = x`.
- Renvoyer une liste interne sans copie.


In [ ]:
# BUG classique : le getter se rappelle lui-même
class Compte:
    def __init__(self, solde):
        self._solde = solde

    @property
    def solde(self):
        return self.solde              # ❌ devrait être self._solde

try:
    Compte(1).solde
except RecursionError:
    print("RecursionError : le getter s'appelle en boucle")
